In [0]:
from pyspark.sql.functions import col, current_timestamp, from_utc_timestamp, max as spark_max, regexp_replace, try_to_timestamp, lit, timestamp_seconds

def ingest_vendor_bronze(vendor_name, file_path, table_name, date_column_name="last_updated", delimiter=",", date_format="yyyy-MM-dd HH:mm:ss"):
    layer_name = "bronze"
    
    try:
        print(f"--- Starting Bronze Ingestion for: {vendor_name} ---")
        
        watermark_df = spark.sql(f"""
            SELECT last_watermark_value 
            FROM inlap.control.watermark_table 
            WHERE source_vendor = '{vendor_name}'
        """)
        last_watermark = watermark_df.collect()[0]["last_watermark_value"]
        
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("delimiter", delimiter)
            .load(file_path)
        )
        df = df.select([col(c).alias(c.lower()) for c in df.columns])
        date_type = dict(df.dtypes).get(date_column_name)

        # 3. Normalize the vendor date column to a timestamp
        if date_type in {"int", "bigint", "integer", "long", "smallint", "tinyint"}:
            df = df.withColumn(date_column_name, timestamp_seconds(col(date_column_name).cast("bigint")))
        else:
            df = df.withColumn(date_column_name, try_to_timestamp(col(date_column_name).cast("string"), lit(date_format)))
                
        # (Optional safety measure): Filter out rows where the date failed to parse and became null
        # df = df.filter(col(date_column_name).isNotNull())
        
        incremental_df = df.filter(col(date_column_name) > last_watermark)
        
        if incremental_df.isEmpty():
            print(f"No new records found for {vendor_name}. Skipping write.")
            return
            
        new_watermark = incremental_df.agg(spark_max(date_column_name)).collect()[0][0]
        new_watermark_str = str(new_watermark)
        
        incremental_df = (
            incremental_df.withColumn(
                "_ingestion_timestamp",
                from_utc_timestamp(current_timestamp(), "Asia/Kolkata"),
            )
            .withColumn("source_file", col("_metadata.file_path"))
            .withColumn("source_name", regexp_replace(col("_metadata.file_name"), "\\.csv$", ""))
        )
        
        if spark.catalog.tableExists(table_name):
            (
                incremental_df.write.format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(table_name)
            )
        else:
            (
                incremental_df.write.format("delta")
                .mode("overwrite")
                .option("mergeSchema", "true")
                .option("path", f"abfss://datalake@attinlapsa.dfs.core.windows.net/bronze/{vendor_name}/")
                .saveAsTable(table_name)
            )
            
        spark.sql(f"""
            UPDATE inlap.control.watermark_table 
            SET last_watermark_value = '{new_watermark_str}',
                last_run_timestamp = current_timestamp(),
                status = 'SUCCESS'
            WHERE source_vendor = '{vendor_name}'
        """)
        
        spark.sql(f"""
            INSERT INTO inlap.control.audit_log
            VALUES (
                '{vendor_name}',
                '{layer_name}',
                current_timestamp(),
                'SUCCESS',
                CAST(NULL AS BIGINT),
                CAST(NULL AS BIGINT),
                CAST(NULL AS BIGINT)
            )
        """)
        print(f"Successfully processed and logged {vendor_name}.")

    except Exception as e:
        print(f"Pipeline FAILED for {vendor_name}. Error: {str(e)}")
        spark.sql(f"""
            INSERT INTO inlap.control.audit_log
            VALUES (
                '{vendor_name}',
                '{layer_name}',
                current_timestamp(),
                'FAILED',
                CAST(NULL AS BIGINT),
                CAST(NULL AS BIGINT),
                CAST(NULL AS BIGINT)
            )
        """)
        raise e

In [0]:
# from pyspark.sql.functions import col, current_timestamp, from_utc_timestamp, max as spark_max, regexp_replace, to_timestamp
# from delta.tables import DeltaTable

# def ingest_vendor_bronze(vendor_name, file_path, table_name, date_column_name="last_updated",delimiter=",",date_format="yyyy-MM-dd HH:mm:ss"):
#     layer_name = "bronze"
    
#     try:
#         print(f"--- Starting Bronze Ingestion for: {vendor_name} ---")
        
#         # 1. Fetch current watermark value for this specific vendor
#         watermark_df = spark.sql(f"""
#             SELECT last_watermark_value 
#             FROM inlap.control.watermark_table 
#             WHERE source_vendor = '{vendor_name}'
#         """)
#         last_watermark = watermark_df.collect()[0]["last_watermark_value"]
#         print(f"Current watermark cutoff: {last_watermark}")
        
#         # 2. Read raw CSV source data
#         df = (
#             spark.read.format("csv")
#             .option("header", "true")
#             .option("inferSchema", "true")
#             .option("delimiter", delimiter)
#             .load(file_path)
#         )
        
#         # 3. Dynamically cast whichever date column the vendor uses
#         df = df.withColumn(date_column_name, to_timestamp(col(date_column_name), date_format))
        
#         # 4. Apply Watermark Filter (Incremental Load)
#         incremental_df = df.filter(col(date_column_name) > last_watermark)
        
#         # 5. Check if new records exist
#         if incremental_df.isEmpty():
#             print(f"No new records found for {vendor_name}. Skipping write.")
#             return
            
#         # Find max timestamp in this new batch for the next watermark
#         new_watermark = incremental_df.agg(spark_max(date_column_name)).collect()[0][0]
#         new_watermark_str = str(new_watermark)
        
#         # 6. Apply standard metadata transformations
#         incremental_df = (
#             incremental_df.withColumn(
#                 "_ingestion_timestamp",
#                 from_utc_timestamp(current_timestamp(), "Asia/Kolkata"),
#             )
#             .withColumn("source_file", col("_metadata.file_path"))
#             .withColumn("source_name", regexp_replace(col("_metadata.file_name"), "\\.csv$", ""))
#         )
        
#         # 7. Write to Delta Table using Catalog check
#         if spark.catalog.tableExists(table_name):
#             print("Table exists in catalog. Appending incremental batch...")
#             (
#                 incremental_df.write.format("delta")
#                 .mode("append")
#                 .option("mergeSchema", "true")
#                 .saveAsTable(table_name)
#             )
#         else:
#             print("Table does not exist. Initializing Delta table...")
#             (
#                 incremental_df.write.format("delta")
#                 .mode("overwrite")
#                 .option("mergeSchema", "true")
#                 .option("path", f"abfss://datalake@attinlapsa.dfs.core.windows.net/bronze/{vendor_name}/")  # Clean dedicated storage path
#                 .saveAsTable(table_name)
#             )
            
#         # 8. Update Watermark Table on Success
#         spark.sql(f"""
#             UPDATE inlap.control.watermark_table 
#             SET last_watermark_value = '{new_watermark_str}',
#                 last_run_timestamp = current_timestamp(),
#                 status = 'SUCCESS'
#             WHERE source_vendor = '{vendor_name}'
#         """)
#         print(f"Watermark advanced to: {new_watermark_str}")
        
#         # 9. Log Success Audit Entry
#         spark.sql(f"""
#             INSERT INTO inlap.control.audit_log 
#             VALUES ('{vendor_name}', '{layer_name}', current_timestamp(), 'SUCCESS')
#         """)
#         print(f"Audit log recorded successfully for {vendor_name}.")

#     except Exception as e:
#         print(f"Pipeline FAILED for {vendor_name}. Error: {str(e)}")
#         spark.sql(f"""
#             INSERT INTO inlap.control.audit_log 
#             VALUES ('{vendor_name}', '{layer_name}', current_timestamp(), 'FAILED')
#         """)
#         raise e

In [0]:
# from pyspark.sql.functions import col, current_timestamp, from_utc_timestamp, max as spark_max, regexp_replace, to_timestamp
# from delta.tables import DeltaTable

# def ingest_vendor_bronze(vendor_name, file_path, table_name, date_column_name="last_updated"):
#     layer_name = "bronze"
    
#     try:
#         print(f"--- Starting Bronze Ingestion for: {vendor_name} ---")
        
#         # 1. Fetch current watermark value
#         watermark_df = spark.sql(f"""
#             SELECT last_watermark_value 
#             FROM inlap.control.watermark_table 
#             WHERE source_vendor = '{vendor_name}'
#         """)
#         last_watermark = watermark_df.collect()[0]["last_watermark_value"]
#         print(f"Current watermark cutoff: {last_watermark}")
        
#         # 2. Read raw CSV source data
#         df = (
#             spark.read.format("csv")
#             .option("header", "true")
#             .option("inferSchema", "true")
#             .load(file_path)
#         )
        
#         # 3. Cast date field to standard timestamp
#         df = df.withColumn(date_column_name, to_timestamp(col(date_column_name), "MM/dd/yyyy HH:mm:ss"))
        
#         # 4. Apply Watermark Filter (Incremental Load)
#         incremental_df = df.filter(col(date_column_name) > last_watermark)
        
#         # 5. Check if new records exist
#         if incremental_df.isEmpty():
#             print(f"No new records found for {vendor_name}. Skipping write.")
#             return
            
#         # Find max timestamp in this new batch for the next watermark
#         new_watermark = incremental_df.agg(spark_max(date_column_name)).collect()[0][0]
#         new_watermark_str = str(new_watermark)
        
#         # 6. Apply metadata transformations
#         incremental_df = (
#             incremental_df.withColumn(
#                 "_ingestion_timestamp",
#                 from_utc_timestamp(current_timestamp(), "Asia/Kolkata"),
#             )
#             .withColumn("source_file", col("_metadata.file_path"))
#             .withColumn("source_name", regexp_replace(col("_metadata.file_name"), "\\.csv$", ""))
#         )
        
#         # 7. Write to Delta Table (Append if exists, Overwrite if first time)
#         if DeltaTable.isDeltaTable(spark, table_path):
#             print("Table exists. Appending incremental batch...")
#             (
#                 incremental_df.write.format("delta")
#                 .mode("append")
#                 .option("mergeSchema", "true")
#                 .saveAsTable(table_name)
#             )
#         else:
#             print("Table does not exist. Initializing Delta table...")
#             (
#                 incremental_df.write.format("delta")
#                 .mode("overwrite")
#                 .option("mergeSchema", "true")
#                 .option("path", table_path)
#                 .saveAsTable(table_name)
#             )
            
#         # 8. Update Watermark Table on Success
#         spark.sql(f"""
#             UPDATE inlap.control.watermark_table 
#             SET last_watermark_value = '{new_watermark_str}',
#                 last_run_timestamp = current_timestamp(),
#                 status = 'SUCCESS'
#             WHERE source_vendor = '{vendor_name}'
#         """)
#         print(f"Watermark advanced to: {new_watermark_str}")
        
#         # 9. Log Success Audit Entry
#         spark.sql(f"""
#             INSERT INTO inlap.control.audit_log 
#             VALUES ('{vendor_name}', '{layer_name}', current_timestamp(), 'SUCCESS')
#         """)
#         print(f"Audit log recorded successfully for {vendor_name}.")

#     except Exception as e:
#         # --- EXCEPTION & FAILURE HANDLING (Step 7) ---
#         print(f"Pipeline FAILED for {vendor_name}. Error: {str(e)}")
        
#         # Log FAILED status in audit log
#         spark.sql(f"""
#             INSERT INTO inlap.control.audit_log 
#             VALUES ('{vendor_name}', '{layer_name}', current_timestamp(), 'FAILED')
#         """)
        
#         # Crucial: Watermark is NOT updated, ensuring the failed window is retried next time
#         raise e

In [0]:
# from pyspark.sql.functions import col, current_timestamp, from_utc_timestamp, max as spark_max, regexp_replace, to_timestamp
# from delta.tables import DeltaTable

# def ingest_vendor_bronze(vendor_name, file_path, table_name, date_column_name="last_updated"):
#     print(f"--- Starting Bronze Ingestion for: {vendor_name} ---")
    
#     # 1. Fetch current watermark value
#     watermark_df = spark.sql(f"""
#         SELECT last_watermark_value 
#         FROM inlap.control.watermark_table 
#         WHERE source_vendor = '{vendor_name}'
#     """)
#     last_watermark = watermark_df.collect()[0]["last_watermark_value"]
#     print(f"Current watermark cutoff: {last_watermark}")
    
#     # 2. Read raw CSV source data
#     df = (
#         spark.read.format("csv")
#         .option("header", "true")
#         .option("inferSchema", "true")
#         .load(file_path)
#     )
    
#     # 3. Cast date field to standard timestamp
#     df = df.withColumn(date_column_name, to_timestamp(col(date_column_name), "MM/dd/yyyy HH:mm:ss"))
    
#     # 4. Apply Watermark Filter (Incremental Load)
#     incremental_df = df.filter(col(date_column_name) > last_watermark)
    
#     # 5. Check if new records exist
#     if incremental_df.isEmpty():
#         print(f"No new records found for {vendor_name}. Skipping write.")
#         return
        
#     # Find max timestamp in this new batch for the next watermark
#     new_watermark = incremental_df.agg(spark_max(date_column_name)).collect()[0][0]
#     new_watermark_str = str(new_watermark)
    
#     # 6. Apply metadata transformations
#     incremental_df = (
#         incremental_df.withColumn(
#             "_ingestion_timestamp",
#             from_utc_timestamp(current_timestamp(), "Asia/Kolkata"),
#         )
#         .withColumn("source_file", col("_metadata.file_path"))
#         .withColumn("source_name", regexp_replace(col("_metadata.file_name"), "\\.csv$", ""))
#     )
    
#     # 7. Write to Delta Table (Append if exists, Overwrite if first time)
#     if DeltaTable.isDeltaTable(spark, table_path):
#         print("Table exists. Appending incremental batch...")
#         (
#             incremental_df.write.format("delta")
#             .mode("append")
#             .option("mergeSchema", "true")
#             .saveAsTable(table_name)
#         )
#     else:
#         print("Table does not exist. Initializing Delta table...")
#         (
#             incremental_df.write.format("delta")
#             .mode("overwrite")
#             .option("mergeSchema", "true")
#             .option("path", table_path)
#             .saveAsTable(table_name)
#         )
        
#     # 8. Update Watermark Table on Success
#     spark.sql(f"""
#         UPDATE inlap.control.watermark_table 
#         SET last_watermark_value = '{new_watermark_str}',
#             last_run_timestamp = current_timestamp(),
#             status = 'SUCCESS'
#         WHERE source_vendor = '{vendor_name}'
#     """)
#     print(f"Watermark advanced to: {new_watermark_str}")
    
#     # 9. Log Audit Entry
#     spark.sql(f"""
#         INSERT INTO inlap.control.audit_log 
#         VALUES ('{vendor_name}', 'bronze', current_timestamp(), 'SUCCESS')
#     """)
#     print(f"Audit log recorded successfully for {vendor_name}.")